Data Preprocessing
------------------------------------------------------------------
1. Define your class labels

2. Convert numerical to categorical

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load your dataset
df = pd.read_csv('your_dataset.csv')

# Define a function to assign labels based on your criteria
def assign_strength_class(strength_value):
    """
    Convert numerical strength to categorical class
    Modify this function based on YOUR dataset and class definitions
    """
    if strength_value < 20:  # Adjust thresholds for YOUR data
        return 1
    elif strength_value < 30:
        return 2
    elif strength_value < 40:
        return 3
    elif strength_value < 50:
        return 4
    else:
        return 5

# Apply the function to create a new column
df['strength_class'] = df['strength'].apply(assign_strength_class)

# View the result
print(df[['strength', 'strength_class']].head(10))

# Save the converted dataset
df.to_csv('converted_dataset.csv', index=False)
print("Converted dataset saved as 'converted_dataset.csv'")

3. Analyse Class Distribution

In [ ]:
# Plot class distribution
plt.figure(figsize=(10, 6))
class_counts = df['strength_class'].value_counts().sort_index()

# Bar chart
plt.subplot(1, 2, 1)
class_counts.plot(kind='bar', color='steelblue', edgecolor='black')
plt.xlabel('Strength Class')
plt.ylabel('Number of Samples')
plt.title('Class Distribution')
plt.xticks(rotation=0)

# Add value labels on bars
for i, v in enumerate(class_counts):
    plt.text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
plt.subplot(1, 2, 2)
plt.pie(class_counts, labels=class_counts.index, autopct='%1.1f%%', 
        startangle=90, colors=['#ff9999','#66b3ff','#99ff99','#ffcc99','#c2c2f0'])
plt.title('Class Distribution (%)')

plt.tight_layout()
plt.show()

# Check for class imbalance
print("\nClass Distribution Summary:")
print(class_counts)
min_class = class_counts.min()
max_class = class_counts.max()
imbalance_ratio = max_class / min_class
print(f"Smallest class: {min_class} samples")
print(f"Largest class: {max_class} samples")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")

if imbalance_ratio > 2:
    print(" Warning: Significant class imbalance detected!")
    print("   Consider balancing techniques in future modeling stages.")
else:
    print("Classes are reasonably balanced.")

Feature Engineering
-------------------------------------------------
1. Simplify the 'Age' Feature

In [ ]:
# Load your converted dataset
df = pd.read_csv('converted_dataset.csv')

# Examine the age feature
print("Original age values:")
print(df['age'].value_counts().sort_index())
print(f"Unique age values: {df['age'].nunique()}")

# Simplify age by converting to categorical
# Based on typical concrete testing ages (1, 3, 7, 28, 90 days)
def simplify_age(age):
    """
    Group ages into meaningful categories
    Modify based on YOUR domain
    """
    if age <= 3:
        return 1  # Early age
    elif age <= 7:
        return 2  # Standard early
    elif age <= 28:
        return 3  # Standard curing
    elif age <= 90:
        return 4  # Long-term
    else:
        return 5  # Extended

df['age_category'] = df['age'].apply(simplify_age)

# Compare original vs simplified
print("\nSimplified age categories:")
print(df['age_category'].value_counts().sort_index())

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['age'].value_counts().sort_index().plot(kind='bar', ax=axes[0], title='Original Age Distribution')
df['age_category'].value_counts().sort_index().plot(kind='bar', ax=axes[1], title='Simplified Age Categories')
plt.tight_layout()
plt.show()

2. Normalize Numerical Features

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Identify features to normalize (exclude target and simplified features)
features_to_normalize = ['cement', 'slag', 'ash', 'water', 'superplastic', 
                         'coarseagg', 'fineagg']  # Adjust for YOUR dataset

# Check if all features exist
available_features = [f for f in features_to_normalize if f in df.columns]
print(f"Normalizing: {available_features}")

# Initialize scaler
scaler = MinMaxScaler()

# Fit and transform
df_normalized = df.copy()
df_normalized[available_features] = scaler.fit_transform(df[available_features])

# Compare before and after
print("\nBefore Normalization:")
print(df[available_features].describe())

print("\nAfter Normalization:")
print(df_normalized[available_features].describe())

# Visualize before/after for one feature
feature_example = available_features[0]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df[feature_example].hist(bins=30, ax=axes[0], edgecolor='black')
axes[0].set_title(f'Original {feature_example}')
df_normalized[feature_example].hist(bins=30, ax=axes[1], edgecolor='black')
axes[1].set_title(f'Normalized {feature_example}')
plt.tight_layout()
plt.show()

# Save intermediate result
df_normalized.to_csv('normalized_dataset.csv', index=False)

3. Create Composite Features

Based on EDA findings, create new features that capture relationships between variables:

In [ ]:
# Load normalized data
df = pd.read_csv('normalized_dataset.csv')

# Create composite features based on EDA insights
# Example from cement dataset: features that showed strong relationships

# 1. Cement-Slag interaction (both are cementitious materials)
df['cement_slag'] = df['cement'] * df['slag']

# 2. Cement-Ash interaction (supplementary materials)
df['cement_ash'] = df['cement'] * df['ash']

# 3. Water-Fine Aggregate ratio (important for workability)
df['water_fineagg'] = df['water'] / (df['fineagg'] + 0.001)  # Avoid division by zero

# 4. Ash + Superplasticizer (chemical interaction)
df['ash_superplastic'] = df['ash'] + df['superplastic']

# View new features
print("New composite features:")
print(df[['cement_slag', 'cement_ash', 'water_fineagg', 'ash_superplastic']].head())

# Check correlation with target
print("\nCorrelation with target class:")
for feature in ['cement_slag', 'cement_ash', 'water_fineagg', 'ash_superplastic']:
    corr = df[feature].corr(df['strength_class'])
    print(f"{feature}: {corr:.3f}")

4. Combine All Features

In [ ]:
# Select final feature set
final_features = available_features + ['age_category'] + \
                 ['cement_slag', 'cement_ash', 'water_fineagg', 'ash_superplastic']

# Add target class
final_columns = final_features + ['strength_class']

# Create final dataframe
df_final = df[final_columns].copy()

# Save
df_final.to_csv('features_dataset.csv', index=False)
print(f"Final features dataset saved with {len(final_features)} features + target")
print(f"Shape: {df_final.shape}")

3. Feature Selection

In [ ]:
# Load your features dataset
df = pd.read_csv('features_dataset.csv')

# Based on EDA, select the most important features
selected_features = ['cement', 'water', 'superplastic', 'age_category',
                     'cement_slag', 'cement_ash', 'water_fineagg', 'ash_superplastic']

# Verify all features exist
selected_features = [f for f in selected_features if f in df.columns]
print(f"Selected features: {selected_features}")

# Create selected feature dataset
df_selected = df[selected_features + ['strength_class']].copy()

# Save
df_selected.to_csv('selected_features_dataset.csv', index=False)
print(f"Selected features dataset saved with {len(selected_features)} features")
print(f"Shape: {df_selected.shape}")

YOUR feature selection based on EDA

In [ ]:
# Method 1: Correlation-based selection
correlations = df.corr()['target_class'].abs().sort_values(ascending=False)
top_features = correlations[correlations > 0.3].index.tolist()
print(f"Features with |correlation| > 0.3: {top_features}")

# Method 2: Domain knowledge selection
domain_features = ['feature1', 'feature2', 'feature3']  # Replace with YOUR domain choices

# Method 3: Combine both
selected_features = list(set(top_features + domain_features))
selected_features = [f for f in selected_features if f != 'target_class']

print(f"\nFinal selected features ({len(selected_features)}):")
print(selected_features)

# Create final dataset
df_final = df[selected_features + ['target_class']].copy()
df_final.to_csv('final_modeling_dataset.csv', index=False)